In [1]:
import duckdb
import pandas as pd

# Open in read_only mode so background scripts or app servers can still write/query
con = duckdb.connect("../data/processed/analytics.duckdb", read_only=True)

df_preview = con.execute("""
    SELECT 
        customer_id,
        cohort_month,
        event_date,
        tier,
        mrr,
        event_type
    FROM subscription_events
    ORDER BY customer_id, event_date
    LIMIT 10;
""").df()

df_preview

,customer_id,cohort_month,event_date,tier,mrr,event_type
0,CUST-1001,2024-01,2024-01-21,Starter,29.0,new
1,CUST-1001,2024-01,2024-02-20,Starter,29.0,renew
2,CUST-1001,2024-01,2024-03-21,Starter,29.0,renew
3,CUST-1001,2024-01,2024-04-20,Starter,29.0,renew
4,CUST-1001,2024-01,2024-05-20,Starter,29.0,renew
5,CUST-1001,2024-01,2024-06-19,Starter,29.0,renew
6,CUST-1001,2024-01,2024-07-19,Starter,29.0,renew
7,CUST-1001,2024-01,2024-08-18,Starter,29.0,renew
8,CUST-1001,2024-01,2024-09-17,Starter,29.0,renew
9,CUST-1001,2024-01,2024-10-17,Starter,29.0,renew


In [2]:
con.execute("""
    SELECT 
        tier,
        COUNT(*) as total_events,
        ROUND(AVG(mrr), 2) as avg_mrr,
        COUNT(DISTINCT customer_id) as unique_customers
    FROM subscription_events
    GROUP BY tier
    ORDER BY unique_customers DESC;
""").df()

,tier,total_events,avg_mrr,unique_customers
0,Starter,18308,27.34,3367
1,Pro,11194,93.44,2208
2,Enterprise,5158,287.52,872


In [4]:
from pathlib import Path

# 1. Read the production MRR waterfall SQL query
sql_path = Path("../src/queries/mrr_waterfall.sql")
with open(sql_path, "r") as f:
    mrr_query = f.read()

# 2. Execute against your DuckDB table
df_waterfall = con.execute(mrr_query).df()

# 3. View the financial waterfall
df_waterfall

,ACTIVITY_MONTH,TOTAL_NEW_MRR,TOTAL_EXPANSION_MRR,TOTAL_CONTRACTION_MRR,TOTAL_CHURNED_MRR,NET_NEW_MRR,ENDING_MRR
0,2024-01,23110.0,0.0,0.0,29.0,23081.0,23651.0
1,2024-02,25460.0,620.0,200.0,4205.0,21675.0,44215.0
2,2024-03,20710.0,2920.0,810.0,5462.0,17358.0,63015.0
3,2024-04,24540.0,2910.0,680.0,7304.0,19466.0,81580.0
4,2024-05,23190.0,3060.0,820.0,7447.0,17983.0,102269.0
5,2024-06,22560.0,5000.0,890.0,8295.0,18375.0,117938.0
6,2024-07,22510.0,5770.0,830.0,8327.0,19123.0,141589.0
7,2024-08,22520.0,4820.0,1170.0,9285.0,16885.0,157653.0
8,2024-09,22560.0,6860.0,1960.0,8394.0,19066.0,173012.0
9,2024-10,25050.0,9410.0,1700.0,12684.0,20076.0,199857.0
